# 异常

## RejectedOrderError
当订单因各种原因被投资组合模拟系统拒绝时抛出的异常。
```python
class RejectedOrderError(Exception):
    pass
```

# 核心配置

## InitCashMode
初始资金模式
- Auto (0): 自动模式
  - 模拟过程中资金视为无限
  - 模拟结束后设置为实际花费的总金额
- AutoAlign (1): 自动对齐模式  
  - 设置为所有列（资产）总花费金额的统一值

```python
class InitCashModeT(tp.NamedTuple):
    Auto: int = 0
    AutoAlign: int = 1   # 自动对齐模式：设置为所有列总花费金额的对齐值

InitCashMode = InitCashModeT()
```


## CallSeqType
调用序列类型定义，控制投资组合模拟中多个资产（列）的处理顺序。
- Default (0): 默认顺序
  - 按列的自然顺序从左到右处理
- Reversed (1): 反向顺序
  - 按列的相反顺序从右到左处理  
- Random (2): 随机顺序
  - 每个时间步随机打乱处理顺序
  - 消除顺序偏差，提供更公平的资金分配
- Auto (3): 自动顺序
  - 根据订单的价值动态排序处理
  - 卖单优先执行以释放资金供买单使用
  - 实现更智能的资金利用，但可能引入前瞻偏差

```python
class CallSeqTypeT(tp.NamedTuple):
    Default: int = 0
    Reversed: int = 1
    Random: int = 2
    Auto: int = 3

CallSeqType = CallSeqTypeT()
```

## AccumulationMode
仓位累积模式类型定义，控制投资组合中仓位的累积行为。
- Disabled (0): 禁用累积
    - 不允许任何形式的仓位累积
    - 每次信号都会完整执行，不考虑现有仓位
    - 适用于简单的买入-持有-卖出策略
- Both (1): 双向累积
    - 允许在现有仓位基础上继续增加或减少
    - 支持分批建仓和分批减仓
- AddOnly (2): 仅增加模式
    - 只允许在现有仓位基础上增加
    - 不允许减少现有仓位
    - 适用于趋势跟踪和动量策略
- RemoveOnly (3): 仅减少模式
    - 只允许减少现有仓位
    - 不允许增加现有仓位
    - 适用于止盈和风险控制策略

```python
class AccumulationModeT(tp.NamedTuple):
    Disabled: int = 0
    Both: int = 1
    AddOnly: int = 2
    RemoveOnly: int = 3

AccumulationMode = AccumulationModeT()
```

## ConflictMode 和 DirectionConflictMode
| 特征 | `ConflictModeT` | `DirectionConflictModeT` |
| :--- | :--- | :--- |
| **处理的冲突类型** | **动作冲突**：<br>`入场(Entry)` vs `出场(Exit)` 信号冲突。 | **方向冲突**：<br>`多头入场(Long Entry)` vs `空头入场(Short Entry)` 信号冲突。 |
| **解决的问题** | “在当前时间点，我应该**开/加仓**还是**平/减仓**？” | “在当前时间点，我应该**开多仓**还是**开空仓**？” |
| **Ignore (忽略)** | 忽略冲突的入场和出场信号，维持仓位不变。 | 忽略冲突的多头和空头入场信号，维持仓位不变。 |
| **Entry / Long (优先)** | **(Entry)**<br>优先执行**入场**信号，忽略出场信号。<br>**效果**: 建立或增加当前方向的仓位。 | **(Long)**<br>优先执行**多头入场**信号，忽略空头入场信号。<br>**效果**: 建立或增加**多头**仓位。 |
| **Exit / Short (优先)** | **(Exit)**<br>优先执行**出场**信号，忽略入场信号。<br>**效果**: 平仓或减少当前方向的仓位。 | **(Short)**<br>优先执行**空头入场**信号，忽略多头入场信号。<br>**效果**: 建立或增加**空头**仓位。 |
| **Adjacent (相邻优先)** | **执行与当前状态“相邻”的操作。**<br>该模式下会同时保留入场和出场信号，最终行为取决于 `accumulate` 参数。 (详见下表) | **执行与当前持仓方向相同的入场信号。**<br>**持多仓时**: 执行多头入场(顺势加仓)。<br>**持空仓时**: 执行空头入场(顺势加仓)。<br>**无仓位时**: 忽略所有信号。 |
| **Opposite (相反优先)** | **执行与当前状态“相反”的操作。**<br>例如，持多仓时优先执行平仓信号，为反转做准备。 | **执行与当前持仓方向相反的入场信号。**<br>**持多仓时**: 执行空头入场(逆势反转)。<br>**持空仓时**: 执行多头入场(逆势反转)。<br>**无仓位时**: 忽略所有信号。 |

| `ConflictMode.Adjacent` 与... | 信号处理 | 最终动作 | 仓位变化 | 核心逻辑 |
| :--- | :--- | :--- | :--- | :--- |
| **`accumulate=False`** <br> (默认) | 冲突发生时，`is_entry` 和 `is_exit` 都被保留为 `True`，但累积模式不允许信号叠加。 | **执行 `Exit` (出场)**，因为在非累积模式下，出场指令优先于入场指令。 | **减少** | **风控优先**: 在考虑增仓前，先执行减仓指令。 |
| **`accumulate=True`** <br> (开启累积) | 冲突发生时，`is_entry` 和 `is_exit` 都被保留为 `True`，且累积模式允许信号叠加。 | **同时执行 `Entry` 和 `Exit`**，一个增仓信号和一个减仓信号的效果相互抵消。 | **不变** | **信号叠加**: 允许正负信号同时作用，最终效果为净变化。 |


```python
class ConflictModeT(tp.NamedTuple):
    Ignore: int = 0
    Entry: int = 1
    Exit: int = 2
    Adjacent: int = 3
    Opposite: int = 4

ConflictMode = ConflictModeT()

class DirectionConflictModeT(tp.NamedTuple):
    Ignore: int = 0
    Long: int = 1
    Short: int = 2
    Adjacent: int = 3
    Opposite: int = 4

DirectionConflictMode = DirectionConflictModeT()
```

## OppositeEntryMode
| 模式 | 触发条件 | 信号处理 | 累积模式影响 | 最终效果 | 适用场景 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Ignore** <br> (忽略) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **完全忽略**反向入场信号<br>保持原有信号不变 | 不影响累积模式 | **保持当前仓位**<br>等待正常退出信号 | **坚持方向策略**：<br>- 趋势跟踪策略<br>- 避免频繁换向<br>- 长期持仓策略 |
| **Close** <br> (直接平仓) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **取消**反向入场信号<br>**生成**对应的平仓信号 | **强制禁用**累积模式<br>`accumulate = Disabled` | **立即平仓**<br>回到空仓状态<br>不建立反向仓位 | **保守策略**：<br>- 风险规避<br>- 不确定时先平仓<br>- 防止过度交易 |
| **CloseReduce** <br> (平仓减少) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **取消**反向入场信号<br>**生成**对应的平仓信号 | **保持**原累积模式<br>不修改 `accumulate` | **平仓或减仓**<br>- `accumulate=False`: 完全平仓<br>- `accumulate=True`: 按信号大小减仓 | **灵活调仓策略**：<br>- 分批平仓<br>- 渐进式退出<br>- 部分获利了结 |
| **Reverse** <br> (完全反转) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **保持**反向入场信号<br>不生成额外的平仓信号 | **强制禁用**累积模式<br>`accumulate = Disabled` | **直接反转**<br>从多头→空头<br>或从空头→多头 | **快速反转策略**：<br>- 趋势反转交易<br>- 快速响应市场<br>- 动量策略 |
| **ReverseReduce** <br> (反转减少) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **保持**反向入场信号<br>不生成额外的平仓信号 | **保持**原累积模式<br>不修改 `accumulate` | **反转或渐进转换**<br>- `accumulate=False`: 直接反转<br>- `accumulate=True`: 先减仓再反转 | **渐进反转策略**：<br>- 分批方向转换<br>- 降低转换冲击<br>- 灵活仓位管理 |

| 反向入场模式 | `accumulate=False` | `accumulate=True` | 累积模式变化 |
| :--- | :--- | :--- | :--- |
| **Ignore** | 忽略反向信号，保持当前仓位 | 忽略反向信号，保持当前仓位 | **不变** |
| **Close** | 立即完全平仓，回到空仓 | 立即完全平仓，回到空仓 | **强制禁用** → `False` |
| **CloseReduce** | 立即完全平仓，回到空仓 | 按信号大小减少仓位，可能部分平仓 | **保持不变** |
| **Reverse** | 立即完全反转仓位方向 | 立即完全反转仓位方向 | **强制禁用** → `False` |
| **ReverseReduce** | 立即完全反转仓位方向 | 先减少当前仓位，完全平仓后建立反向仓位 | **保持不变** |


```python
class OppositeEntryModeT(tp.NamedTuple):
    Ignore: int = 0
    Close: int = 1
    CloseReduce: int = 2
    Reverse: int = 3
    ReverseReduce: int = 4

OppositeEntryMode = OppositeEntryModeT()
```

## StopEntryPrice
止损入场价格类型，义了在设置止损订单时使用哪种价格作为初始止损价格的基准。
- ValPrice (0): 资产估值价格
  - 使用资产的当前估值价格作为止损基准
  - 通常是最新的市场价格或理论价值
- Price (1): 默认价格
  - 使用系统默认的价格作为基准
  - 通常是当前K线的某个标准价格
  - 最常用的选择，适合大多数场景
- FillPrice (2): 实际成交价格
  - 使用订单的实际成交价格（已包含滑点影响）
  - 反映真实的交易成本
  - 适用于精确的风险控制和盈亏计算
- Close (3): 收盘价格
  - 使用K线的收盘价作为基准
  - 提供稳定的价格参考

```python
class StopEntryPriceT(tp.NamedTuple):
    ValPrice: int = 0
    Price: int = 1
    FillPrice: int = 2
    Close: int = 3

StopEntryPrice = StopEntryPriceT()
```

## StopExitPrice
止损退出价格类型，定义了当止损信号触发时，使用哪种价格执行平仓操作。
- StopLimit (0): 止损限价单
  - 以止损触发价格作为限价单执行
  - 如果止损之前已被触发，使用下一K线的开盘价
  - 不应用用户自定义的滑点设置
  - 提供价格保护但可能无法成交
- SStopMarket (1): 止损市价单
  - 以止损触发价格作为市价单执行
  - 如果止损之前已被触发，使用下一K线的开盘价
  - 应用用户自定义的滑点设置
  - 确保成交但价格可能不理想
- SPrice (2): 默认价格
  - 使用系统默认价格执行止损
  - 应用用户自定义的滑点设置
  - 最常用的选择
- SClose (3): 收盘价格
  - 使用K线收盘价执行止损
  - 应用用户自定义的滑点设置
  - 适用于较长周期的策略

```python
class StopExitPriceT(tp.NamedTuple):
    StopLimit: int = 0
    StopMarket: int = 1
    Price: int = 2
    Close: int = 3

StopExitPrice = StopExitPriceT()
```

## StopExitMode
止损退出模式类型，定义当止损信号触发时如何处理当前仓位。
- Close (0): 直接平仓
    - 止损触发时立即平掉当前全部仓位
    - 回到空仓状态，不建立新的仓位
- CloseReduce (1): 平仓或减少
    - 禁用累积模式时：直接平掉当前仓位
    - 启用累积模式时：按照止损信号大小减少仓位
- Reverse (2): 完全反转
    - 平掉当前仓位并建立等量的反向仓位
    - 从多头直接转为等量空头（或相反）
    - 适用于认为市场将反向运行的策略
- ReverseReduce (3): 反转或减少
    - 禁用累积模式时：完全反转仓位方向
    - 启用累积模式时：先减少当前仓位，完全平仓后建立反向仓位
    - 在累积模式下提供渐进式的反转能力
    - 结合了反转策略和渐进式调整的优点

```python
class StopExitModeT(tp.NamedTuple):
    Close: int = 0
    CloseReduce: int = 1
    Reverse: int = 2
    ReverseReduce: int = 3

StopExitMode = StopExitModeT()
```

## StopUpdateMode
止损更新模式类型，定义当建立新仓位时如何处理已有的止损设置。
- Keep (0): 保持原有止损
    - 保留旧的止损设置不变
    - 新的仓位继续使用原有的止损水平
    - 适用于希望维持一致止损策略的场景
    - 确保止损策略的连续性
- Override (1): 条件覆盖模式
    - 只有当新的止损不为NaN时才覆盖旧设置
    - 如果新止损为NaN，则保持原有止损
    - 提供了智能的止损更新机制
    - 避免因数据问题导致止损失效
- OverrideNaN (2): 强制覆盖模式
    - 无论新止损是否为NaN都覆盖旧设置
    - 即使新止损为NaN也会替换原有止损
    - 可能导致止损保护失效，需要谨慎使用
    - 适用于需要完全重置止损的场景

```python
class StopUpdateModeT(tp.NamedTuple):
    Keep: int = 0
    Override: int = 1
    OverrideNaN: int = 2

StopUpdateMode = StopUpdateModeT()
```

# 订单和交易

## SizeType
订单大小类型定义，定义不同的订单大小指定方式。
- Amount (0): 绝对数量模式
    - 直接指定要交易的资产数量（如股数、手数等）
- Value (1): 价值金额模式
    - 指定要交易的资产价值金额
- Percent (2): 可用资源百分比模式
    - 基于当前可用资源的百分比（注意：不是仓位价值的百分比！）
    - 买入时：基于 OrderContext.cash_now 的百分比
    - 卖出时：基于 OrderContext.position_now 的百分比  
    - 做空时：基于 OrderContext.free_cash_now 的百分比
    - 反转仓位时：同时考虑position_now和free_cash_now
    - 自动考虑手续费和滑点限制
- TargetAmount (3): 目标数量模式
    - 指定最终要持有的资产目标数量（=目标仓位）
    - 使用 OrderContext.position_now 获取当前仓位
    - 自动计算需要交易的数量，转换为Amount模式
    - 适用于仓位再平衡策略
- TargetValue (4): 目标价值模式
    - 指定最终要持有的资产目标价值
    - 使用 OrderContext.val_price_now 获取当前资产价值
    - 转换为TargetAmount模式执行
    - 适用于基于价值的仓位管理
- TargetPercent (5): 目标百分比模式
    - 指定目标仓位占总投资组合价值的百分比
    - 使用 OrderContext.value_now 获取当前总价值
    - 转换为TargetValue模式执行
    - 适用于动态再平衡和风险平价策略

```python
class SizeTypeT(tp.NamedTuple):
    Amount: int = 0
    Value: int = 1
    Percent: int = 2
    TargetAmount: int = 3
    TargetValue: int = 4
    TargetPercent: int = 5

SizeType = SizeTypeT()
```

## Direction
仓位方向类型，定义投资组合中允许的仓位方向。
- LongOnly (0): 仅多头方向
  - 只允许买入持有（做多）操作
  - 不能进行卖空操作
  - 适用于传统的长期投资策略
  - 风险相对较低，适合保守投资者
- ShortOnly (1): 仅空头方向
  - 只允许卖空（做空）操作
  - 不能进行买入持有操作
  - 适用于熊市或特定的做空策略
  - 风险较高，需要专业知识和技能
- Both (2): 双向交易
  - 同时允许多头和空头操作
  - 可以根据市场条件灵活调整仓位方向
  - 适用于对冲策略和市场中性策略
  - 提供最大的策略灵活性

```python
class DirectionT(tp.NamedTuple):
    LongOnly: int = 0
    ShortOnly: int = 1
    Both: int = 2

Direction = DirectionT()
```

## OrderStatus
订单状态类型定义，定义订单在投资组合模拟系统中的最终执行状态。
- Filled (0): 已成交状态
  - 订单已成功执行并产生了实际的交易记录
  - 资金和仓位已发生相应变化
  - 这是正常交易的期望状态
- Ignored (1): 已忽略状态
  - 订单被系统主动忽略，通常是由于策略规则限制
  - 例如：信号冲突时选择忽略、累积模式下的重复信号等
  - 不会产生任何资金或仓位变化
- Rejected (2): 已拒绝状态
  - 订单因各种限制条件无法执行而被拒绝
  - 如资金不足、仓位不足、订单大小不符合要求等
  - 具体拒绝原因可通过OrderStatusInfo查看

```python
class OrderStatusT(tp.NamedTuple):
    Filled: int = 0
    Ignored: int = 1
    Rejected: int = 2

OrderStatus = OrderStatusT()
```

## OrderSide
订单方向类型，定义订单的交易方向，区分买入和卖出操作。
- Buy (0): 买入方向
    - 购买资产，使用现金换取资产
    - 对多头仓位：增加持仓数量
    - 对空头仓位：减少空头数量（买入平仓）
    - 资金流向：现金减少，资产增加
- Sell (1): 卖出方向  
    - 出售资产，使用资产换取现金
    - 对多头仓位：减少持仓数量（卖出平仓）
    - 对空头仓位：增加空头数量（卖空开仓）
    - 资金流向：资产减少，现金增加
```python
class OrderSideT(tp.NamedTuple):
    Buy: int = 0
    Sell: int = 1

OrderSide = OrderSideT()
```

## OrderStatusInfo
订单状态详细信息类型，提供订单被忽略或拒绝的具体原因。
- SizeNaN (0): 订单大小为NaN
  - 订单大小参数包含无效数值
  - 通常由计算错误或数据问题导致
  - 需要检查size参数的计算逻辑
- PriceNaN (1): 订单价格为NaN
  - 订单执行价格无效
  - 可能是价格数据缺失或计算错误
  - 需要检查价格数据的完整性
- ValPriceNaN (2): 估值价格为NaN
  - 资产估值价格无效
  - 影响价值计算和仓位评估
  - 需要检查估值数据源
- ValueNaN (3): 价值为NaN
  - 计算得出的订单价值无效
  - 通常是价格或数量计算错误的结果
- ValueZeroNeg (4): 价值为零或负数
  - 订单价值不合理（零或负数）
  - 可能是价格或数量设置错误
- SizeZero (5): 订单大小为零
  - 计算得出的订单大小为零
  - 通常发生在目标仓位等于当前仓位时
- NoCashShort (6): 做空时现金不足
  - 做空订单所需的保证金不足
  - 需要增加现金或减少订单大小
- NoCashLong (7): 做多时现金不足  
  - 买入订单所需的资金不足
  - 最常见的拒绝原因之一
- NoOpenPosition (8): 没有可平仓的持仓
  - 尝试卖出但没有相应的多头仓位
  - 或尝试买入平仓但没有空头仓位
- MaxSizeExceeded (9): 超过最大订单大小限制
  - 订单大小超过了预设的最大值
  - 用于风险控制和仓位管理
- RandomEvent (10): 随机拒绝事件
  - 模拟真实市场中的随机拒绝情况
  - 通过reject_prob参数控制
- CantCoverFees (11): 无法支付手续费
  - 剩余资金不足以支付交易手续费
  - 需要考虑手续费对小额交易的影响
- MinSizeNotReached (12): 未达到最小订单大小
  - 订单大小低于预设的最小值
  - 用于避免过小的无意义交易
- PartialFill (13): 部分成交
  - 订单只能部分执行
  - 通常发生在资金或仓位不足时

```python
class OrderStatusInfoT(tp.NamedTuple):
    SizeNaN: int = 0
    PriceNaN: int = 1
    ValPriceNaN: int = 2
    ValueNaN: int = 3
    ValueZeroNeg: int = 4
    SizeZero: int = 5
    NoCashShort: int = 6
    NoCashLong: int = 7
    NoOpenPosition: int = 8
    MaxSizeExceeded: int = 9
    RandomEvent: int = 10
    CantCoverFees: int = 11
    MinSizeNotReached: int = 12
    PartialFill: int = 13

OrderStatusInfo = OrderStatusInfoT()
```

## TradeDirection
交易方向类型，定义交易记录中的方向属性。
- Long (0): 多头交易
  - 从买入开仓开始，到卖出平仓结束的完整交易
  - 盈利来源于价格上涨
  - 传统的"低买高卖"交易模式
- Short (1): 空头交易
  - 从卖空开仓开始，到买入平仓结束的完整交易
  - 盈利来源于价格下跌
  - "高卖低买"的交易模式

```python
class TradeDirectionT(tp.NamedTuple):
    Long: int = 0
    Short: int = 1

TradeDirection = TradeDirectionT()
```

## TradeStatus
交易状态类型，定义交易记录的当前状态，区分正在进行的交易和已完成的交易。
- Open (0): 开放状态（未平仓）
    - 交易仍在进行中，持有未平仓的仓位
    - 盈亏为浮动盈亏，随市价变化
    - 交易的最终结果尚未确定
    
- Closed (1): 关闭状态（已平仓）
    - 交易已完全完成，所有仓位已平掉
    - 盈亏为已实现盈亏，结果确定
    - 可以计算准确的投资回报率

```python
class TradeStatusT(tp.NamedTuple):
    Open: int = 0
    Closed: int = 1

TradeStatus = TradeStatusT()
```

## TradesType
交易记录类型,定义不同类型的交易记录分析.
- EntryTrades (0): 入场交易分析
    - 以入场信号为起点的交易分析
    - 关注入场时机、入场价格、入场后的表现
    - 适用于分析入场策略的有效性
    - 每个入场信号产生一个交易记录
- ExitTrades (1): 出场交易分析  
    - 以出场信号为终点的交易分析
    - 关注出场时机、出场价格、持仓期间的表现
    - 适用于分析出场策略的有效性
    - 每个出场信号产生一个交易记录
- Positions (2): 仓位生命周期分析
    - 完整的仓位从建立到清空的分析
    - 包含完整的买入-持有-卖出周期
    - 适用于分析整体仓位管理效果
    - 每个完整的仓位周期产生一个记录

```python
class TradesTypeT(tp.NamedTuple):
    EntryTrades: int = 0
    ExitTrades: int = 1
    Positions: int = 2

TradesType = TradesTypeT()
```

# 状态和上下文

## ProcessOrderState
订单处理状态数据结构,记录订单处理前后的完整投资组合状态信息.

```python
class ProcessOrderState(tp.NamedTuple):
    cash: float      # 现金余额：当前列或现金共享组的现金总额
    position: float  # 持仓数量：当前列的资产持仓数量（正数多头，负数空头）
    debt: float      # 做空债务：当前列做空操作产生的债务总额
    free_cash: float # 可用现金：当前列或现金共享组的可用于交易的现金
    val_price: float # 估值价格：当前列资产的估值价格（用于价值计算）
    value: float     # 总价值：当前列或现金共享组的总价值（现金+持仓价值-债务）
    oidx: int        # 订单索引：对应的订单记录在order_records数组中的索引位置
    lidx: int        # 日志索引：对应的日志记录在log_records数组中的索引位置
```

## ExecuteOrderState
订单执行状态数据结构,记录订单执行完成后的核心状态信息,这是ProcessOrderState的简化版本.

```python
class ExecuteOrderState(tp.NamedTuple):
    cash: float      # 现金余额：订单执行后的现金总额
    position: float  # 持仓数量：订单执行后的持仓数量
    debt: float      # 做空债务：订单执行后的债务总额
    free_cash: float # 可用现金：订单执行后的可用现金
```

## SimulationContext
模拟上下文数据结构, 包含了整个模拟过程中所有必需的配置参数、状态信息和记录数组.
```python
class SimulationContext(tp.NamedTuple):
    # 模拟配置参数
    target_shape: tp.Shape          # 目标形状：(行数, 列数) 
    group_lens: tp.Array1d          # 每组的列数数组
    init_cash: tp.Array1d           # 初始资金数组
    cash_sharing: bool              # 是否启用现金共享
    call_seq: tp.Optional[tp.Array2d] # 调用序列矩阵
    segment_mask: tp.ArrayLike      # 段执行掩码
    call_pre_segment: bool          # 是否调用段前函数
    call_post_segment: bool         # 是否调用段后函数
    close: tp.ArrayLike            # 收盘价数据
    ffill_val_price: bool          # 是否前向填充估值价格
    update_value: bool             # 是否在订单后更新价值
    fill_pos_record: bool          # 是否填充仓位记录
    flex_2d: bool                  # 是否使用灵活的二维索引
    
    # 记录存储数组
    order_records: tp.RecordArray   # 订单记录数组
    log_records: tp.RecordArray     # 日志记录数组
    
    # 最新状态数组
    last_cash: tp.Array1d           # 最新现金余额
    last_position: tp.Array1d       # 最新持仓数量
    last_debt: tp.Array1d           # 最新做空债务
    last_free_cash: tp.Array1d      # 最新可用现金
    last_val_price: tp.Array1d      # 最新估值价格
    last_value: tp.Array1d          # 最新组合价值
    second_last_value: tp.Array1d   # 次新组合价值
    last_return: tp.Array1d         # 最新收益率
    last_oidx: tp.Array1d           # 最新订单记录索引
    last_lidx: tp.Array1d           # 最新日志记录索引
    last_pos_record: tp.RecordArray # 最新仓位记录
```

## GroupContext
资产组上下文数据结构, 表示当前处理的资产组的上下文信息.

资产组是一组相关的列（资产），它们可能共享现金或具有其他关联关系。

该上下文包含了SimulationContext的所有字段，并添加了描述当前组的特定信息

```python
class GroupContext(tp.NamedTuple):
    # 继承自SimulationContext的所有字段
    target_shape: tp.Shape          # 模拟目标形状
    group_lens: tp.Array1d          # 每组列数
    init_cash: tp.Array1d           # 初始资金
    cash_sharing: bool              # 现金共享标志
    call_seq: tp.Optional[tp.Array2d] # 调用序列
    segment_mask: tp.ArrayLike      # 段掩码
    call_pre_segment: bool          # 调用段前函数标志
    call_post_segment: bool         # 调用段后函数标志
    close: tp.ArrayLike            # 收盘价数据
    ffill_val_price: bool          # 前向填充估值价格标志
    update_value: bool             # 更新价值标志
    fill_pos_record: bool          # 填充仓位记录标志
    flex_2d: bool                  # 灵活二维索引标志
    order_records: tp.RecordArray   # 订单记录数组
    log_records: tp.RecordArray     # 日志记录数组
    last_cash: tp.Array1d           # 最新现金状态
    last_position: tp.Array1d       # 最新仓位状态
    last_debt: tp.Array1d           # 最新债务状态
    last_free_cash: tp.Array1d      # 最新可用现金
    last_val_price: tp.Array1d      # 最新估值价格
    last_value: tp.Array1d          # 最新组合价值
    second_last_value: tp.Array1d   # 次新组合价值
    last_return: tp.Array1d         # 最新收益率
    last_oidx: tp.Array1d           # 最新订单索引
    last_lidx: tp.Array1d           # 最新日志索引
    last_pos_record: tp.RecordArray # 最新仓位记录
    
    # GroupContext特有字段
    group: int        # 当前组的索引
    group_len: int    # 当前组中的列数
    from_col: int     # 当前组第一列的索引
    to_col: int       # 当前组最后一列的索引+1
```

## RowContext
行上下文数据结构, 表示当前时间步（行）的上下文信息。

包含了SimulationContext的所有字段，并添加了当前行的特定信息。

```python
class RowContext(tp.NamedTuple):
    # 继承自SimulationContext的所有字段
    target_shape: tp.Shape          # 模拟目标形状
    group_lens: tp.Array1d          # 每组列数
    init_cash: tp.Array1d           # 初始资金
    cash_sharing: bool              # 现金共享标志
    call_seq: tp.Optional[tp.Array2d] # 调用序列
    segment_mask: tp.ArrayLike      # 段掩码
    call_pre_segment: bool          # 调用段前函数标志
    call_post_segment: bool         # 调用段后函数标志
    close: tp.ArrayLike            # 收盘价数据
    ffill_val_price: bool          # 前向填充估值价格标志
    update_value: bool             # 更新价值标志
    fill_pos_record: bool          # 填充仓位记录标志
    flex_2d: bool                  # 灵活二维索引标志
    order_records: tp.RecordArray   # 订单记录数组
    log_records: tp.RecordArray     # 日志记录数组
    last_cash: tp.Array1d           # 最新现金状态
    last_position: tp.Array1d       # 最新仓位状态
    last_debt: tp.Array1d           # 最新债务状态
    last_free_cash: tp.Array1d      # 最新可用现金
    last_val_price: tp.Array1d      # 最新估值价格
    last_value: tp.Array1d          # 最新组合价值
    second_last_value: tp.Array1d   # 次新组合价值
    last_return: tp.Array1d         # 最新收益率
    last_oidx: tp.Array1d           # 最新订单索引
    last_lidx: tp.Array1d           # 最新日志索引
    last_pos_record: tp.RecordArray # 最新仓位记录
    
    # RowContext特有字段
    i: int                          # 当前行（时间步）索引
```

## SegmentContext
段上下文数据结构, 表示一个段的上下文信息.

段是组和行的交集，定义了在同一组和行内元素的处理方式和顺序。

包含了多个上下文的所有字段，并添加了描述当前段的特定字段。

```python
总资产列表: [股票0, 股票1, 股票2, 股票3, 股票4, 股票5, 股票6, 股票7, 股票8]
             ↑                    ↑                    ↑
            组0 (3只)            组1 (2只)            组2 (4只)
           from_col=0            from_col=3           from_col=5
           to_col=3              to_col=5             to_col=9
```

```python
class SegmentContext(tp.NamedTuple):
    # 继承字段（来自 SimulationContext）
    # 模拟配置参数
    target_shape: tp.Shape          # 模拟目标形状
    group_lens: tp.Array1d          # 每组列数
    init_cash: tp.Array1d           # 初始资金
    cash_sharing: bool              # 现金共享标志
    call_seq: tp.Optional[tp.Array2d] # 调用序列
    segment_mask: tp.ArrayLike      # 段掩码
    call_pre_segment: bool          # 调用段前函数标志
    call_post_segment: bool         # 调用段后函数标志
    # 市场数据
    close: tp.ArrayLike            # 收盘价数据
    ffill_val_price: bool          # 前向填充估值价格标志
    update_value: bool             # 更新价值标志
    fill_pos_record: bool          # 填充仓位记录标志
    flex_2d: bool                  # 灵活二维索引标志
    # 记录存储
    order_records: tp.RecordArray   # 订单记录数组
    log_records: tp.RecordArray     # 日志记录数组
    # 最新状态数组
    last_cash: tp.Array1d           # 最新现金状态
    last_position: tp.Array1d       # 最新仓位状态
    last_debt: tp.Array1d           # 最新债务状态
    last_free_cash: tp.Array1d      # 最新可用现金
    last_val_price: tp.Array1d      # 最新估值价格
    last_value: tp.Array1d          # 最新组合价值
    second_last_value: tp.Array1d   # 次新组合价值
    last_return: tp.Array1d         # 最新收益率
    last_oidx: tp.Array1d           # 最新订单索引
    last_lidx: tp.Array1d           # 最新日志索引
    last_pos_record: tp.RecordArray # 最新仓位记录
    # 继承字段（来自 GroupContext）
    group: int                      # 当前组索引
    group_len: int                  # 当前组大小
    from_col: int                   # 组起始列索引
    to_col: int                     # 组结束列索引+1
    # 继承字段（来自 RowContext）
    i: int                          # 当前行索引
    # SegmentContext特有字段
    call_seq_now: tp.Optional[tp.Array1d] # 当前段内的调用序列
```

## OrderContext
订单上下文数据结构, 表示当前订单执行时的完整上下文信息。

这是最详细的上下文，包含了 SegmentContext 的所有字段，并添加了描述当前状态的特定字段.
```python
class OrderContext(tp.NamedTuple):
    # 继承自上级上下文的所有字段
    target_shape: tp.Shape          # 模拟目标形状
    group_lens: tp.Array1d          # 每组列数
    init_cash: tp.Array1d           # 初始资金
    cash_sharing: bool              # 现金共享标志
    call_seq: tp.Optional[tp.Array2d] # 调用序列
    segment_mask: tp.ArrayLike      # 段掩码
    call_pre_segment: bool          # 调用段前函数标志
    call_post_segment: bool         # 调用段后函数标志
    close: tp.ArrayLike            # 收盘价数据
    ffill_val_price: bool          # 前向填充估值价格标志
    update_value: bool             # 更新价值标志
    fill_pos_record: bool          # 填充仓位记录标志
    flex_2d: bool                  # 灵活二维索引标志
    order_records: tp.RecordArray   # 订单记录数组
    log_records: tp.RecordArray     # 日志记录数组
    last_cash: tp.Array1d           # 最新现金状态
    last_position: tp.Array1d       # 最新仓位状态
    last_debt: tp.Array1d           # 最新债务状态
    last_free_cash: tp.Array1d      # 最新可用现金
    last_val_price: tp.Array1d      # 最新估值价格
    last_value: tp.Array1d          # 最新组合价值
    second_last_value: tp.Array1d   # 次新组合价值
    last_return: tp.Array1d         # 最新收益率
    last_oidx: tp.Array1d           # 最新订单索引
    last_lidx: tp.Array1d           # 最新日志索引
    last_pos_record: tp.RecordArray # 最新仓位记录
    group: int                      # 当前组索引
    group_len: int                  # 当前组大小
    from_col: int                   # 组起始列索引
    to_col: int                     # 组结束列索引+1
    i: int                          # 当前行索引
    call_seq_now: tp.Optional[tp.Array1d] # 当前段调用序列
    
    # OrderContext特有字段 - 当前状态
    col: int                        # 当前列（资产）索引
    call_idx: int                   # 当前调用索引
    cash_now: float                 # 当前现金余额
    position_now: float             # 当前持仓数量
    debt_now: float                 # 当前做空债务
    free_cash_now: float            # 当前可用现金
    val_price_now: float            # 当前估值价格
    value_now: float                # 当前组合价值
    return_now: float               # 当前收益率
    pos_record_now: tp.Record       # 当前仓位记录
```

## PostOrderContext
订单后上下文数据结构, 表示订单处理完成后的上下文信息。

包含了OrderContext的所有字段，并添加了订单执行结果和执行前状态的字段.
```python
class PostOrderContext(tp.NamedTuple):
    # 继承自上级上下文的所有字段（省略重复字段注释）
    target_shape: tp.Shape
    group_lens: tp.Array1d
    init_cash: tp.Array1d
    cash_sharing: bool
    call_seq: tp.Optional[tp.Array2d]
    segment_mask: tp.ArrayLike
    call_pre_segment: bool
    call_post_segment: bool
    close: tp.ArrayLike
    ffill_val_price: bool
    update_value: bool
    fill_pos_record: bool
    flex_2d: bool
    order_records: tp.RecordArray
    log_records: tp.RecordArray
    last_cash: tp.Array1d
    last_position: tp.Array1d
    last_debt: tp.Array1d
    last_free_cash: tp.Array1d
    last_val_price: tp.Array1d
    last_value: tp.Array1d
    second_last_value: tp.Array1d
    last_return: tp.Array1d
    last_oidx: tp.Array1d
    last_lidx: tp.Array1d
    last_pos_record: tp.RecordArray
    group: int
    group_len: int
    from_col: int
    to_col: int
    i: int
    call_seq_now: tp.Optional[tp.Array1d]
    col: int
    call_idx: int
    
    # 执行前状态字段
    cash_before: float          # 执行前现金余额
    position_before: float      # 执行前持仓数量
    debt_before: float          # 执行前做空债务
    free_cash_before: float     # 执行前可用现金
    val_price_before: float     # 执行前估值价格
    value_before: float         # 执行前组合价值
    
    # 执行结果字段
    order_result: "OrderResult" # 订单执行结果
    
    # 执行后状态字段
    cash_now: float             # 执行后现金余额
    position_now: float         # 执行后持仓数量
    debt_now: float             # 执行后做空债务
    free_cash_now: float        # 执行后可用现金
    val_price_now: float        # 执行后估值价格
    value_now: float            # 执行后组合价值
    return_now: float           # 执行后收益率
    pos_record_now: tp.Record   # 执行后仓位记录
```

## FlexOrderContext
灵活订单上下文数据结构, 表示灵活订单执行时的上下文信息.

与OrderContext不同，FlexOrderContext不绑定到特定的列，而是提供更灵活的订单生成方式。

包含了SegmentContext的所有字段，并添加了当前调用索引。
```python
class FlexOrderContext(tp.NamedTuple):
    # 继承自上级上下文的所有字段
    target_shape: tp.Shape          # 模拟目标形状
    group_lens: tp.Array1d          # 每组列数
    init_cash: tp.Array1d           # 初始资金
    cash_sharing: bool              # 现金共享标志
    call_seq: tp.Optional[tp.Array2d] # 调用序列
    segment_mask: tp.ArrayLike      # 段掩码
    call_pre_segment: bool          # 调用段前函数标志
    call_post_segment: bool         # 调用段后函数标志
    close: tp.ArrayLike            # 收盘价数据
    ffill_val_price: bool          # 前向填充估值价格标志
    update_value: bool             # 更新价值标志
    fill_pos_record: bool          # 填充仓位记录标志
    flex_2d: bool                  # 灵活二维索引标志
    order_records: tp.RecordArray   # 订单记录数组
    log_records: tp.RecordArray     # 日志记录数组
    last_cash: tp.Array1d           # 最新现金状态
    last_position: tp.Array1d       # 最新仓位状态
    last_debt: tp.Array1d           # 最新债务状态
    last_free_cash: tp.Array1d      # 最新可用现金
    last_val_price: tp.Array1d      # 最新估值价格
    last_value: tp.Array1d          # 最新组合价值
    second_last_value: tp.Array1d   # 次新组合价值
    last_return: tp.Array1d         # 最新收益率
    last_oidx: tp.Array1d           # 最新订单索引
    last_lidx: tp.Array1d           # 最新日志索引
    last_pos_record: tp.RecordArray # 最新仓位记录
    group: int                      # 当前组索引
    group_len: int                  # 当前组大小
    from_col: int                   # 组起始列索引
    to_col: int                     # 组结束列索引+1
    i: int                          # 当前行索引
    
    # FlexOrderContext特有字段
    call_seq_now: None              # 无调用序列（灵活模式）
    call_idx: int                   # 当前调用索引
```

# 订单相关

## Order
订单定义数据结构, 包含了执行一个订单所需的所有参数和控制选项.
```python
class Order(tp.NamedTuple):
    size: float = np.inf              # 订单大小：要交易的数量或金额
    price: float = np.inf             # 订单价格：每单位的交易价格
    size_type: int = SizeType.Amount  # 大小类型：订单大小的解释方式
    direction: int = Direction.Both   # 允许方向：订单允许的交易方向
    fees: float = 0.0                # 手续费率：按订单价值的百分比收费
    fixed_fees: float = 0.0          # 固定手续费：每笔订单的固定费用
    slippage: float = 0.0            # 滑点率：价格滑动的百分比
    min_size: float = 0.0            # 最小大小：订单的最小允许大小
    max_size: float = np.inf         # 最大大小：订单的最大允许大小
    size_granularity: float = np.nan # 大小粒度：订单大小的最小调整单位
    reject_prob: float = 0.0         # 拒绝概率：随机拒绝订单的概率
    lock_cash: bool = False          # 锁定现金：做空时是否锁定现金
    allow_partial: bool = True       # 允许部分成交：是否接受部分填充
    raise_reject: bool = False       # 拒绝异常：拒绝时是否抛出异常
    log: bool = False               # 日志记录：是否记录此订单的详细日志
```

## NoOrder
空订单实例，表示不应该被处理的订单，用于跳过特定的交易时点.

```python
NoOrder = Order(
    size=np.nan,
    price=np.nan,
    size_type=-1,
    direction=-1,
    fees=np.nan,
    fixed_fees=np.nan,
    slippage=np.nan,
    min_size=np.nan,
    max_size=np.nan,
    size_granularity=np.nan,
    reject_prob=np.nan,
    lock_cash=False,
    allow_partial=False,
    raise_reject=False,
    log=False
)
```

## OrderResult
订单执行结果数据结构, 记录单个订单执行完成后的结果信息.
```python
class OrderResult(tp.NamedTuple):
    size: float    # 实际成交大小
    price: float   # 实际成交价格（含滑点调整）
    fees: float    # 实际支付的总手续费
    side: int      # 实际执行的订单方向（OrderSide枚举）
    status: int    # 订单执行状态（OrderStatus枚举）
    status_info: int # 订单状态详细信息（OrderStatusInfo枚举）
```

# 调整上下文

## AdjustSLContext
止损调整上下文数据结构, 用于止损订单调整函数的上下文信息，特别是跟踪止损（Trailing Stop Loss）的调整.
```python
class AdjustSLContext(tp.NamedTuple):
    i: int                    # 当前行索引：当前时间步的索引位置，范围[0, target_shape[0])
    col: int                  # 当前列索引：当前资产列的索引，范围[0, target_shape[1])，且在[from_col, to_col)内
    position_now: float       # 当前持仓数量：当前时间点的资产持仓数量（正数多头，负数空头）
    val_price_now: float      # 当前估值价格：当前时间点的资产估值价格，用于止损计算
    init_i: int               # 初始止损行索引：止损首次设置时的时间步索引，保持不变
    init_price: float         # 初始止损价格：止损首次设置时的价格基准，保持不变
    curr_i: int               # 当前止损行索引：最近一次止损更新的时间步索引，随价格更新而变化
    curr_price: float         # 当前止损价格：最近一次更新的止损价格，在跟踪止损中随价格上涨而上移
    curr_stop: float          # 当前止损值：当前有效的止损价格，可由调整函数修改
    curr_trail: bool          # 当前跟踪标志：当前是否为跟踪止损模式，可由调整函数修改
```

## AdjustTPContext
止盈调整上下文数据结构, 用于止盈订单调整函数的上下文信息.
```python
class AdjustTPContext(tp.NamedTuple):
    i: int                    # 当前行索引：当前时间步的索引位置，范围[0, target_shape[0])
    col: int                  # 当前列索引：当前资产列的索引，范围[0, target_shape[1])，且在[from_col, to_col)内
    position_now: float       # 当前持仓数量：当前时间点的资产持仓数量（正数多头，负数空头）
    val_price_now: float      # 当前估值价格：当前时间点的资产估值价格，用于止盈计算
    init_i: int               # 初始止盈行索引：止盈首次设置时的时间步索引，保持不变
    init_price: float         # 初始止盈价格：止盈首次设置时的价格基准，保持不变
    curr_stop: float          # 当前止盈值：当前有效的止盈价格，可由调整函数修改
```

## SignalContext
信号生成上下文数据结构, 用于信号生成函数的上下文信息.

该上下文提供了生成交易信号所需的所有当前状态信息.
```python
class SignalContext(tp.NamedTuple):
    i: int                    # 当前行索引：当前时间步的索引位置，范围[0, target_shape[0])
    col: int                  # 当前列索引：当前资产列的索引，范围[0, target_shape[1])，且在[from_col, to_col)内
    position_now: float       # 当前持仓数量：当前时间点的资产持仓数量（正数多头，负数空头）
    val_price_now: float      # 当前估值价格：当前时间点的资产估值价格，用于信号计算
    flex_2d: bool             # 灵活二维索引标志：是否使用灵活的二维索引选择，参见vectorbt.base.reshape_fns.flex_select_auto_nb
```

# 记录数据类型

## order_dt
订单记录数据类型，定义了订单记录的完整结构化数据格式.
```python
order_dt = np.dtype([
    ('id', np.int64),        # 订单唯一标识符：每个订单的唯一ID，用于订单追踪和引用
    ('col', np.int64),       # 列索引（资产索引）：订单对应的资产列索引，标识交易的具体资产
    ('idx', np.int64),       # 时间索引（行索引）：订单执行的时间步索引，标识交易的具体时间点
    ('size', np.float64),    # 订单大小（实际成交数量）：订单实际成交的资产数量，正数买入，负数卖出
    ('price', np.float64),   # 订单价格（实际成交价格）：订单实际成交的价格，已包含滑点调整
    ('fees', np.float64),    # 手续费金额：订单执行产生的总手续费，包括固定费用和比例费用
    ('side', np.int64),      # 订单方向（买入/卖出）：订单的交易方向，0表示买入，1表示卖出
], align=True)
```

## trade_dt
交易记录数据类型，定义了完整交易生命周期的结构化数据格式

```python
_trade_fields = [
    ('id', np.int64),           # 交易唯一标识符：每个交易的全局唯一ID，从0开始递增，用于交易追踪和绩效分析
    ('col', np.int64),          # 列索引（资产索引）：指示交易所属的资产列，范围[0, 资产数量-1]，用于多资产组合的交易分组
    ('size', np.float64),       # 交易大小（持仓数量）：交易的最大持仓数量，正数表示多头交易，负数表示空头交易，反映交易的规模和风险暴露
    ('entry_idx', np.int64),    # 入场时间索引：交易开始（首次建仓）的时间点，对应价格数据的行索引，用于计算持仓时间
    ('entry_price', np.float64), # 平均入场价格：所有入场订单的加权平均价格，考虑了分批建仓的情况，用于计算交易盈亏的成本基础
    ('entry_fees', np.float64), # 入场总手续费：建仓过程中产生的所有手续费总和，包括多次加仓的累计费用，影响交易净收益的成本因素
    ('exit_idx', np.int64),     # 出场时间索引：交易结束（完全平仓）的时间点，对于未平仓交易为-1，用于计算实际持仓期间
    ('exit_price', np.float64), # 平均出场价格：所有出场订单的加权平均价格，对于未平仓交易，使用当前估值价格，用于计算最终交易盈亏
    ('exit_fees', np.float64),  # 出场总手续费：平仓过程中产生的所有手续费总和，包括分批平仓的累计费用，对于未平仓交易可能为0
    ('pnl', np.float64),        # 盈亏金额：交易的绝对盈亏金额（包含手续费），计算公式：(exit_price - entry_price) * size - entry_fees - exit_fees，正数表示盈利，负数表示亏损
    ('return', np.float64),     # 收益率：交易的相对收益率，计算公式：pnl / (entry_price * abs(size) + entry_fees)，用于标准化的绩效比较
    ('direction', np.int64),    # 交易方向（多头/空头）：使用TradeDirection枚举值，0=Long（多头），1=Short（空头），区分交易的基本策略方向，用于方向性绩效分析
    ('status', np.int64),       # 交易状态（开放/关闭）：使用TradeStatus枚举值，0=Open（开放），1=Closed（关闭），区分活跃交易和历史交易，影响盈亏计算的方式
    ('parent_id', np.int64)     # 父交易ID（用于关联分析）：用于关联相关交易的父级标识，支持复杂的交易关系建模，便于分层交易分析
]
trade_dt = np.dtype(_trade_fields, align=True)
```

## log_dt
日志记录数据类型，定义了订单执行过程的完整状态记录格式

```python
_log_fields = [
    # 基础标识字段
    ('id', np.int64),                    # 日志记录唯一标识符：全局递增的记录ID，用于日志追踪和调试
    ('group', np.int64),                 # 资产组索引：用于现金共享分组，标识日志记录所属的资产组
    ('col', np.int64),                   # 列索引（资产索引）：指示相关资产，范围[0, 资产数量-1]
    ('idx', np.int64),                   # 时间索引（行索引）：记录发生的时间点，对应价格数据的行索引
    
    # 执行前状态字段
    ('cash', np.float64),                # 执行前现金余额：订单执行前的现金总额，用于状态对比分析
    ('position', np.float64),            # 执行前持仓数量：订单执行前的资产持仓数量，正数多头，负数空头
    ('debt', np.float64),                # 执行前做空债务：订单执行前的做空债务总额，影响可用资金计算
    ('free_cash', np.float64),           # 执行前可用现金：订单执行前的可用于交易的现金，考虑债务和锁定资金
    ('val_price', np.float64),           # 执行前资产估值价格：订单执行前的资产估值价格，用于价值计算
    ('value', np.float64),               # 执行前组合总价值：订单执行前的组合总价值，现金+持仓价值-债务
    
    # 订单请求参数字段
    ('req_size', np.float64),            # 请求的订单大小：策略请求的交易数量或金额，可能因各种限制而调整
    ('req_price', np.float64),           # 请求的订单价格：策略请求的交易价格，可能因滑点而调整
    ('req_size_type', np.int64),         # 请求的订单大小类型：使用SizeType枚举，定义订单大小的解释方式
    ('req_direction', np.int64),         # 请求的交易方向：使用Direction枚举，定义允许的交易方向
    ('req_fees', np.float64),            # 请求的手续费率：按订单价值的百分比收费，影响交易成本
    ('req_fixed_fees', np.float64),      # 请求的固定手续费：每笔订单的固定费用，与订单大小无关
    ('req_slippage', np.float64),        # 请求的滑点率：价格滑动的百分比，影响实际成交价格
    ('req_min_size', np.float64),        # 请求的最小订单大小：订单的最小允许大小，小于此值可能被拒绝
    ('req_max_size', np.float64),        # 请求的最大订单大小：订单的最大允许大小，超过此值可能被拒绝
    ('req_size_granularity', np.float64), # 请求的订单大小粒度：订单大小的最小调整单位，用于精度控制
    ('req_reject_prob', np.float64),     # 请求的拒绝概率：随机拒绝订单的概率，用于模拟市场拒绝
    ('req_lock_cash', np.bool_),         # 请求的现金锁定标志：做空时是否锁定现金，影响资金管理
    ('req_allow_partial', np.bool_),     # 请求的部分成交允许标志：是否接受部分填充，影响订单执行策略
    ('req_raise_reject', np.bool_),      # 请求的拒绝异常标志：拒绝时是否抛出异常，影响错误处理
    ('req_log', np.bool_),               # 请求的日志记录标志：是否记录此订单的详细日志，影响调试信息
    
    # 执行后状态字段
    ('new_cash', np.float64),            # 执行后现金余额：订单执行后的现金总额，反映资金变化
    ('new_position', np.float64),        # 执行后持仓数量：订单执行后的资产持仓数量，反映仓位变化
    ('new_debt', np.float64),            # 执行后做空债务：订单执行后的做空债务总额，反映债务变化
    ('new_free_cash', np.float64),       # 执行后可用现金：订单执行后的可用于交易的现金，反映可用资金变化
    ('new_val_price', np.float64),       # 执行后资产估值价格：订单执行后的资产估值价格，可能因价格更新而变化
    ('new_value', np.float64),           # 执行后组合总价值：订单执行后的组合总价值，反映整体价值变化
    
    # 执行结果字段
    ('res_size', np.float64),            # 实际执行的订单大小：订单实际成交的数量，可能因资金限制而小于请求大小
    ('res_price', np.float64),           # 实际执行的订单价格：订单实际成交的价格，包含滑点调整
    ('res_fees', np.float64),            # 实际产生的手续费：订单执行产生的实际手续费，包括比例费用和固定费用
    ('res_side', np.int64),              # 实际执行的订单方向：使用OrderSide枚举，0=Buy（买入），1=Sell（卖出）
    ('res_status', np.int64),            # 订单执行状态：使用OrderStatus枚举，0=Filled（已成交），1=Ignored（已忽略），2=Rejected（已拒绝）
    ('res_status_info', np.int64),       # 订单状态详细信息：使用OrderStatusInfo枚举，提供被忽略或拒绝的具体原因
    ('order_id', np.int64)               # 关联的订单记录ID：对应的订单记录在order_records数组中的索引位置，用于关联分析
]
log_dt = np.dtype(_log_fields, align=True)
```